In [5]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (20,10)

from pypfopt.expected_returns import mean_historical_return,ema_historical_return,capm_return
from pypfopt.risk_models import CovarianceShrinkage ,risk_matrix
from pypfopt.efficient_frontier import EfficientFrontier
from pypfopt.black_litterman import BlackLittermanModel ,market_implied_prior_returns,market_implied_risk_aversion


In [6]:
symbols= ['RELIANCE.NS','HDFCBANK.NS','ICICIBANK.NS','HDFC.NS','INFY.NS','ITC.NS','TCS.NS','HINDUNILVR.NS','SBIN.NS','BHARTIARTL.NS']
sd = yf.download(symbols, interval='1d')['Close']
sd.reset_index(inplace=True)
# rename column ['Date'] to ['date']
sd.rename(columns={'^NSEI':'Nifty50'}, inplace=True)
# reaarange columns
sd = sd[['Date','RELIANCE.NS','HDFCBANK.NS','ICICIBANK.NS','HDFCBANK.NS','INFY.NS','ITC.NS','TCS.NS','HINDUNILVR.NS','SBIN.NS','BHARTIARTL.NS','Nifty50']]
symbols.remove('^NSEI')
symbols.append('Nifty50')
sd

[*********************100%***********************]  10 of 10 completed

1 Failed download:
- HDFC.NS: No data found, symbol may be delisted


KeyError: "['Nifty50'] not in index"

In [7]:
sd['Date'] = pd.to_datetime(sd['Date'])
sd['Date'] = sd['Date'].dt.strftime('%Y-%m-%d')
# get month and year from8 date
Month = pd.DatetimeIndex(sd['Date']).month
Year = pd.DatetimeIndex(sd['Date']).year
sd['month_year'] = Year.astype(str) + '-' + Month.astype(str)
sd.reset_index(inplace=True, drop=True)
sd

,Date,BHARTIARTL.NS,HDFC.NS,HDFCBANK.NS,HINDUNILVR.NS,ICICIBANK.NS,INFY.NS,ITC.NS,RELIANCE.NS,SBIN.NS,TCS.NS,month_year
0,1996-01-01,NaN,NaN,2.980000,61.805000,NaN,0.796679,5.583333,14.691803,18.823240,NaN,1996-1
1,1996-01-02,NaN,NaN,2.975000,62.465000,NaN,0.793457,5.372222,14.577553,18.224106,NaN,1996-1
2,1996-01-03,NaN,NaN,2.985000,62.095001,NaN,0.798828,5.200000,14.688232,17.738192,NaN,1996-1
3,1996-01-04,NaN,NaN,2.965000,62.099998,NaN,0.793554,5.297777,14.552561,17.676863,NaN,1996-1
4,1996-01-05,NaN,NaN,2.960000,62.000000,NaN,0.784179,5.202222,14.452592,17.577793,NaN,1996-1
...,...,...,...,...,...,...,...,...,...,...,...,...
7182,2024-08-08,1451.800049,NaN,1642.699951,2733.199951,1164.599976,1743.150024,494.750000,2898.250000,808.049988,4172.549805,2024-8
7183,2024-08-09,1464.099976,NaN,1650.199951,2747.199951,1171.599976,1770.750000,495.899994,2948.600098,824.299988,4228.750000,2024-8
7184,2003-04-14,NaN,NaN,23.965000,142.399994,NaN,41.458591,NaN,NaN,27.192270,38.387501,2003-4
7185,2004-04-26,NaN,NaN,37.985001,151.600006,NaN,82.924217,NaN,NaN,NaN,38.387501,2004-4


In [8]:
# get first and last day of month for each month in the data
months = sd.groupby('month_year')['Date'].agg([ 'last'])
months.sort_values(by='last', inplace=True)
months

,last
month_year,
1996-1,1996-01-31
1996-2,1996-02-29
1996-3,1996-03-29
1996-4,1996-04-30
1996-5,1996-05-31
...,...
2024-4,2024-04-30
2024-5,2024-05-31
2024-6,2024-06-28


In [9]:
sd1 = sd.copy()
sd = sd[sd['Date'].isin(months['last'])]
inde= sd[sd['month_year'] == '2009-1'].index[0]
sd = sd[sd['Date'] >= sd['Date'][inde]]
sd1 = sd1[sd1['Date'] >= sd['Date'][inde]]
sd1.reset_index(inplace=True, drop=True)
# sd1 = sd

sd.reset_index(inplace=True, drop=True)
# returns = sd[symbols].pct_change()
sd1

,Date,BHARTIARTL.NS,HDFC.NS,HDFCBANK.NS,HINDUNILVR.NS,ICICIBANK.NS,INFY.NS,ITC.NS,RELIANCE.NS,SBIN.NS,TCS.NS,month_year
0,2009-01-30,285.676025,NaN,92.559998,261.649994,75.681816,163.331253,60.033333,302.552704,115.099998,127.900002,2009-1
1,2009-02-02,277.497101,NaN,88.870003,261.899994,70.018181,159.956253,59.400002,292.586456,109.580002,122.824997,2009-2
2,2009-02-03,281.642883,NaN,89.820000,266.100006,71.272728,160.312500,60.299999,298.575348,109.230003,124.962502,2009-2
3,2009-02-04,285.833740,NaN,88.584999,267.899994,70.854546,160.206253,60.299999,298.872498,109.680000,124.662498,2009-2
4,2009-02-05,283.287689,NaN,88.425003,263.399994,71.009087,156.981247,60.433334,294.597992,109.474998,120.724998,2009-2
...,...,...,...,...,...,...,...,...,...,...,...,...
3823,2024-08-05,1465.699951,NaN,1615.750000,2715.899902,1172.599976,1751.900024,486.000000,2894.649902,811.650024,4155.049805,2024-8
3824,2024-08-06,1443.550049,NaN,1601.199951,2750.050049,1166.849976,1751.099976,486.299988,2912.100098,797.700012,4171.200195,2024-8
3825,2024-08-07,1441.750000,NaN,1623.500000,2744.050049,1172.449951,1791.650024,492.649994,2929.649902,808.650024,4200.450195,2024-8
3826,2024-08-08,1451.800049,NaN,1642.699951,2733.199951,1164.599976,1743.150024,494.750000,2898.250000,808.049988,4172.549805,2024-8


In [ ]:
sd

In [ ]:
model = linearregression()

In [ ]:
# # black litterman model -- sample test
# from pypfopt import black_litterman, risk_models
# from pypfopt import BlackLittermanModel, plotting
# from pypfopt import expected_returns
#
# S = risk_models.CovarianceShrinkage(sd[symbols[:-1]]).ledoit_wolf()
# mu = expected_returns.mean_historical_return(sd[symbols[:-1]])
# delta = black_litterman.market_implied_risk_aversion(sd[symbols[:-1]])
# # prior = black_litterman.market_implied_prior_returns(mcaps, delta, cov_matrix)
# prior = black_litterman.market_implied_prior_returns(sd[symbols[:-1]].all(), delta, S, risk_free_rate=0.02)
# bl = BlackLittermanModel(S, mu, delta)
# bl.bl_weights()

In [10]:
# create folder if not exists
import os
if not os.path.exists('Results'):
    os.makedirs('Results')


## Months where returns are negative (all returns are negative,so can't invest)
- 2011-02-28
- 2016-02-29
- 2018-03-28

In [11]:

# create folder if not exists
if not os.path.exists('Results/BlackLittermanModel'):
    os.makedirs('Results/BlackLittermanModel')

# creating a dataframe to store all the results
BlackLittermanModel_results = pd.DataFrame(columns=['returns_method','risk_model','weight_method','portfolio_return','nifty_return','return_months','holding_month','error_dates','no_of_error_dates'])
# BlackLittermanModel_results = pd.read_csv('AllResults.csv')

# creating a list of column names
column = ['Date','month_year','holding_months','returns_months']
column = column + [i + '_weights' for i in symbols[:-1]] + [i + '_pct_change' for i in symbols]

list_of_dates = ['2011-02-28','2016-02-29','2018-03-28']

# parameters
returns_methods = [mean_historical_return,ema_historical_return,capm_return]
risk_models_methods = ['sample_cov','semicovariance','exp_cov','ledoit_wolf','ledoit_wolf_single_factor','ledoit_wolf_constant_correlation','oracle_approximating']
method =['bl_weights']

# get oputput for each method in method list and each risk model in risk_models_methods list and each return in returns list
for returns_method in returns_methods: # for each returns method
    for risk_model in risk_models_methods: # for each risk model
        for weight_mrthod in method: # for each weight method
            storing_weights = pd.DataFrame(columns=column) # create a dataframe to store weights
            for returns_months in range(6,0,-1): # for each return month
                for holding_months in range(6,0,-1):
                    main_df_weights = pd.DataFrame(columns=column) # create a dataframe to store weights
                    dates=[] # create a list to store error dates
                    print('start',returns_method.__name__,risk_model,weight_mrthod,returns_months,holding_months,len(sd)) # print the parameters
                    for ind in range(12,len(sd)-(holding_months)): # for each date in the data
                        if sd['Date'][ind] not in list_of_dates: # if date is not in list of dates
                            df = sd1[(sd1['Date'] <= sd['Date'][ind]) & (sd1['Date'] >= sd['Date'][ind-returns_months])][symbols[:-1]] # get data for returns months
                            try:
                                mu = returns_method(df) # get returns using returns method
                                S = risk_matrix(df, method=risk_model) # get risk matrix using risk model
                                delta = market_implied_risk_aversion(df) # get delta
                                bl = BlackLittermanModel(S, mu, delta) # create black litterman model
                                raw_weights = getattr(bl, weight_mrthod)() # get weights using weight method
                                cleaned_weights = bl.clean_weights() # clean weights
                            except Exception as e:
                                print(returns_method.__name__,risk_model,weight_mrthod,sd['Date'][ind],e) # print error
                                dates.append(sd['Date'][ind]) # append error date to list
                                continue
                            cleaned_weights = pd.DataFrame.from_dict(cleaned_weights, orient='index', columns=['weights']) # convert weights to dataframe
                            cleaned_weights[cleaned_weights['weights'] < 0] = 0 # set negative weights to 0
                            cleaned_weights[cleaned_weights['weights'] > 0] = cleaned_weights[cleaned_weights['weights'] > 0] / cleaned_weights[cleaned_weights['weights'] > 0].sum() # normalize weights
                            returns_df = pd.DataFrame(columns=symbols) # create a dataframe to store returns
                            returns_df.loc[0] = sd[sd['Date'] == sd['Date'][ind]][symbols].iloc[0] # insert startrting prices
                            returns_df.loc[1] = sd[sd['Date'] == sd['Date'][ind+holding_months]][symbols].iloc[0] # insert ending prices
                            main_df_weights.loc[len(main_df_weights)] = [sd['Date'][ind],sd['month_year'][ind],str(returns_months),str(holding_months),] + cleaned_weights['weights'].tolist() + returns_df.pct_change().iloc[1].tolist() # insert data into main dataframe

                    for symbol in symbols[:-1]:
                        main_df_weights[symbol + '_multiply'] = main_df_weights[symbol + '_weights'] * main_df_weights[symbol + '_pct_change'] # multiply weights with returns
                    main_df_weights['portfolio_return'] = main_df_weights[[i + '_multiply' for i in symbols[:-1]]].mean(axis=1) # get portfolio returns


                    main_df_weights['nifty_cummulative_return'] = (main_df_weights['Nifty50_pct_change'] + 1).cumprod() # get cummulative returns
                    main_df_weights['portfolio_cummulative_return'] = (main_df_weights['portfolio_return'] + 1).cumprod() # get cummulative returns

                    """"""
                    #   """ Normalization """
                    main_df_weights['Normalization_portfolio_return'] =main_df_weights['portfolio_return'] / holding_months
                    main_df_weights['Normalization_Nifty50_return'] =main_df_weights['Nifty50_pct_change'] / holding_months
                    main_df_weights['Normalization_portfolio_cummulative_return'] = (main_df_weights['Normalization_portfolio_return'] + 1).cumprod() # get cummulative returns
                    main_df_weights['Normalization_nifty_cummulative_return'] = (main_df_weights['Normalization_Nifty50_return'] + 1).cumprod() # get cummulative returns
                    #
                    """"""

                    storing_weights = pd.concat([storing_weights,main_df_weights]) # concat main dataframe with storing dataframe
                    BlackLittermanModel_results.loc[len(BlackLittermanModel_results)] = [returns_method.__name__,risk_model,weight_mrthod,main_df_weights['portfolio_cummulative_return'].iloc[-1],main_df_weights['nifty_cummulative_return'].iloc[-1],returns_months,holding_months,dates,len(dates)] # insert data into all methods dataframe
            storing_weights.to_csv('Results/BlackLittermanModel/BlackLittermanModel_' + returns_method.__name__ + '_' + risk_model + '_' + weight_mrthod + '.csv',index=False) # save main dataframe
        BlackLittermanModel_results.to_csv('AllResultsBlackLitterman.csv',index=False) # save all methods dataframe

start mean_historical_return sample_cov bl_weights 6 6 188


KeyError: 'Nifty50_pct_change'

In [12]:
# create a folder to store results
if not os.path.exists('Results/EfficientFrontier'):
    os.makedirs('Results/EfficientFrontier')

EfficientFrontier_results = pd.DataFrame(columns=['returns_method','risk_model','weight_method','portfolio_return','nifty_return','return_months','holding_month','error_dates','no_of_error_dates'])
# EfficientFrontier_results = pd.read_csv('AllResults.csv')

# create a list of columns
column = ['Date','month_year','holding_months','returns_months']
column = column + [i + '_weights' for i in symbols[:-1]] + [i + '_pct_change' for i in symbols]

list_of_dates = ['2011-02-28','2016-02-29','2018-03-28']

# create a list of returns methods, risk models and weight methods
returns_methods = [mean_historical_return,ema_historical_return,capm_return]
risk_models_methods = ['sample_cov','semicovariance','exp_cov','ledoit_wolf','ledoit_wolf_single_factor','ledoit_wolf_constant_correlation','oracle_approximating']
method =['max_sharpe','min_volatility','efficient_risk','efficient_return','max_quadratic_utility']

# get oputput for each method in method list and each risk model in risk_models_methods list and each return in returns list
for returns_method in returns_methods: # loop through returns methods
    for risk_model in risk_models_methods: # loop through risk models
        for weight_mrthod in method: # loop through weight methods
            storing_weights = pd.DataFrame(columns=column) # create a storing dataframe
            for returns_months in range(6,0,-1): # loop through returns months
                for holding_months in range(6,0,-1): # loop through holding months
                    main_df_weights = pd.DataFrame(columns=column) # create a main dataframe
                    dates=[] # create a list to store error dates
                    print('start',returns_method.__name__,risk_model,weight_mrthod,returns_months,holding_months,len(sd))
                    for ind in range(12,len(sd)-(holding_months)): # loop through each row in dataframe
                        if sd['Date'][ind] not in list_of_dates: # check if date is not in list of dates
                            # print(sd['Date'][ind-returns_months],sd['Date'][ind])
                            df = sd1[(sd1['Date'] <= sd['Date'][ind]) & (sd1['Date'] >= sd['Date'][ind-returns_months])][symbols[:-1]] # get dataframe for return months
                            try:
                                mu = returns_method(df) # get returns using returns method
                                S = risk_matrix(df, method=risk_model) # get risk matrix using risk model
                                # get weights using max sharpe ratio in 0 to 1 range
                                ef = EfficientFrontier(mu, S) # create an object of EfficientFrontier
                                raw_weights = getattr(ef, weight_mrthod)() # get weights using weight method
                                cleaned_weights = ef.clean_weights() # get cleaned weights
                            except Exception as e:
                                print(returns_method.__name__,risk_model,weight_mrthod,sd['Date'][ind],e) # print error
                                dates.append(sd['Date'][ind])  # append error date to list
                                continue
                            cleaned_weights = pd.DataFrame.from_dict(cleaned_weights, orient='index', columns=['weights']) # convert cleaned weights to dataframe
                            cleaned_weights[cleaned_weights['weights'] < 0] = 0 # set negative weights to 0
                            cleaned_weights[cleaned_weights['weights'] > 0] = cleaned_weights[cleaned_weights['weights'] > 0] / cleaned_weights[cleaned_weights['weights'] > 0].sum() # normalize weights
                            returns_df = pd.DataFrame(columns=symbols) # create a returns dataframe
                            returns_df.loc[0] = sd[sd['Date'] == sd['Date'][ind]][symbols].iloc[0] # get returns starting date
                            returns_df.loc[1] = sd[sd['Date'] == sd['Date'][ind+holding_months]][symbols].iloc[0] # get returns ending date
                            main_df_weights.loc[len(main_df_weights)] = [sd['Date'][ind],sd['month_year'][ind],str(returns_months),str(holding_months),] + cleaned_weights['weights'].tolist() + returns_df.pct_change().iloc[1].tolist() # insert data into main dataframe

                    for symbol in symbols[:-1]:
                        main_df_weights[symbol + '_multiply'] = main_df_weights[symbol + '_weights'] * main_df_weights[symbol + '_pct_change'] # multiply weights with returns
                    main_df_weights['portfolio_return'] = main_df_weights[[i + '_multiply' for i in symbols[:-1]]].mean(axis=1) # get portfolio returns

                    main_df_weights['nifty_cummulative_return'] = (main_df_weights['Nifty50_pct_change'] + 1).cumprod() # get nifty_cummulative_return
                    main_df_weights['portfolio_cummulative_return'] = (main_df_weights['portfolio_return'] + 1).cumprod() # get portfolio_cummulative_return

                    """"""
                    #    """ Normalization """
                    main_df_weights['Normalization_portfolio_return'] =main_df_weights['portfolio_return'] / holding_months
                    main_df_weights['Normalization_Nifty50_return'] =main_df_weights['Nifty50_pct_change'] / holding_months
                    main_df_weights['Normalization_portfolio_cummulative_return'] = (main_df_weights['Normalization_portfolio_return'] + 1).cumprod()
                    main_df_weights['Normalization_nifty_cummulative_return'] = (main_df_weights['Normalization_Nifty50_return'] + 1).cumprod()
                    #
                    """"""
                    storing_weights = pd.concat([storing_weights,main_df_weights],axis=0) # concat main dataframe to storing dataframe

                    EfficientFrontier_results.loc[len(EfficientFrontier_results)] = [returns_method.__name__,risk_model,weight_mrthod,main_df_weights['portfolio_cummulative_return'].iloc[-1],main_df_weights['nifty_cummulative_return'].iloc[-1],returns_months,holding_months,dates,len(dates)]
            storing_weights.to_csv('Results/EfficientFrontier/EfficientFrontier_'+returns_method.__name__+'_'+risk_model+'_'+weight_mrthod+'.csv',index=False) # save storing dataframe to csv
        EfficientFrontier_results.to_csv('AllResults_EfficientFrontier.csv',index=False) # save storing dataframe to csv

start mean_historical_return sample_cov max_sharpe 6 6 188
mean_historical_return sample_cov max_sharpe 2010-01-29 Quadratic form matrices must be symmetric/Hermitian.
mean_historical_return sample_cov max_sharpe 2010-02-26 Quadratic form matrices must be symmetric/Hermitian.
mean_historical_return sample_cov max_sharpe 2010-03-31 Quadratic form matrices must be symmetric/Hermitian.
mean_historical_return sample_cov max_sharpe 2010-04-30 Quadratic form matrices must be symmetric/Hermitian.
mean_historical_return sample_cov max_sharpe 2010-05-31 Quadratic form matrices must be symmetric/Hermitian.
mean_historical_return sample_cov max_sharpe 2010-06-30 Quadratic form matrices must be symmetric/Hermitian.
mean_historical_return sample_cov max_sharpe 2010-07-30 Quadratic form matrices must be symmetric/Hermitian.
mean_historical_return sample_cov max_sharpe 2010-08-31 Quadratic form matrices must be symmetric/Hermitian.
mean_historical_return sample_cov max_sharpe 2010-09-30 Quadratic for

mean_historical_return sample_cov max_sharpe 2016-09-30 Quadratic form matrices must be symmetric/Hermitian.
mean_historical_return sample_cov max_sharpe 2016-10-28 Quadratic form matrices must be symmetric/Hermitian.
mean_historical_return sample_cov max_sharpe 2016-11-30 Quadratic form matrices must be symmetric/Hermitian.
mean_historical_return sample_cov max_sharpe 2016-12-30 Quadratic form matrices must be symmetric/Hermitian.
mean_historical_return sample_cov max_sharpe 2017-01-31 Quadratic form matrices must be symmetric/Hermitian.
mean_historical_return sample_cov max_sharpe 2017-02-28 Quadratic form matrices must be symmetric/Hermitian.
mean_historical_return sample_cov max_sharpe 2017-03-31 Quadratic form matrices must be symmetric/Hermitian.
mean_historical_return sample_cov max_sharpe 2017-04-28 Quadratic form matrices must be symmetric/Hermitian.
mean_historical_return sample_cov max_sharpe 2017-05-31 Quadratic form matrices must be symmetric/Hermitian.
mean_historical_ret

mean_historical_return sample_cov max_sharpe 2023-06-30 Quadratic form matrices must be symmetric/Hermitian.
mean_historical_return sample_cov max_sharpe 2023-07-31 Quadratic form matrices must be symmetric/Hermitian.
mean_historical_return sample_cov max_sharpe 2023-08-31 Quadratic form matrices must be symmetric/Hermitian.
mean_historical_return sample_cov max_sharpe 2023-09-29 Quadratic form matrices must be symmetric/Hermitian.
mean_historical_return sample_cov max_sharpe 2023-10-31 Quadratic form matrices must be symmetric/Hermitian.
mean_historical_return sample_cov max_sharpe 2023-11-30 Quadratic form matrices must be symmetric/Hermitian.
mean_historical_return sample_cov max_sharpe 2023-12-29 Quadratic form matrices must be symmetric/Hermitian.
mean_historical_return sample_cov max_sharpe 2024-01-31 Quadratic form matrices must be symmetric/Hermitian.
mean_historical_return sample_cov max_sharpe 2024-02-29 Quadratic form matrices must be symmetric/Hermitian.


KeyError: 'Nifty50_pct_change'

In [13]:
EfficientFrontier_results

,returns_method,risk_model,weight_method,portfolio_return,nifty_return,return_months,holding_month,error_dates,no_of_error_dates


In [14]:
BlackLittermanModel_results

,returns_method,risk_model,weight_method,portfolio_return,nifty_return,return_months,holding_month,error_dates,no_of_error_dates


In [15]:
import shutil
shutil.make_archive('EfficientFrontier', 'zip', 'Results/EfficientFrontier')
shutil.make_archive('BlackLittermanModel', 'zip', 'Results/BlackLittermanModel')

'C:\\Users\\admin\\BlackLittermanModel.zip'